Generate a figure like figure 8 of Blouin et al. 2019 (Paper III) in which the number abundances of each element relative to calcium are compared to other objects with all objects plotted in a column corresponding to the given element.

In [1]:
from __future__ import print_function

import matplotlib

matplotlib.use('pdf')
savefig=True
    
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
import periodictable as pt

start = time.time()
print(start)
time_string=str(start).split('.')[0]

#from mendeleev import O, Ca, Li, Na, Si, Fe, Mg, He
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs


#print(os.getcwd())

1658327538.5419068
all_wctb
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv


Now I need to choose the elements that will appear in the plot. I could import an element list and check against it for ones that have a match... but I'd rather just manually list out the elements it should be checking for instead

In [2]:
base_el='Ca'
el_list=['Li','Na','Mg','K','Cr','Fe']
#el_list=['Na','K','Li','Cr','Fe','Mg']


el_space=2


ssp=True

In [3]:
figure_output_dir='/Users/BenKaiser/Desktop/'
#figure_output_dir='/Users/BenKaiser/Desktop/z_plots_for_Hollands/'
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/ApJ_reformat/figures'

In [4]:
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [5]:
#wd_abund_file='all_wd_abundances.csv'
#wd_abund_file='20210131_all_wd_abundances.csv'
#wd_abund_file='20210713_all_wd_abundances_bedard_cooling_no_WDs_plot.csv'
#wd_abund_file='20210818_all_wd_abundances_bedard_cooling_newer_Blouin.csv'
#wd_abund_file='20211112_all_wd_abundances_beryllium_objects_partially_added.csv'
#wd_abund_file='20220222_all_wd_abundances_newMC_ages.csv'
#wd_abund_file='20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'
lodders_abund_file='Lodders2020_solarsystem_abundances.csv'
#solar_system_object_file='solar_system_body_abundances.csv'
#solar_system_object_file='solar_system_body_abundances_mgfe_fixed.csv'
#solar_system_object_file='solar_system_body_abundances_sea_added.csv'
#solar_system_object_file='20210713_solar_system_body_abundances_show_name_added.csv'
solar_system_object_file='20210713_solar_system_body_abundances_all_names.csv'
crust_file='20220304_continental_crust_vals_only.csv'

wd_num_abund_file='20220204_beryllium_WD_linear_number_abundances.csv'



In [6]:
if ssp:
    wd_abund_file='20220304_WD_SSP_abundances.csv'
else:
    wd_abund_file='20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'



In [7]:
print(os.getcwd())

print(wd_abund_file)
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
lodders_table=Table.read(lodders_abund_file)
bodies_table=Table.read(solar_system_object_file)
crust_table=Table.read(crust_file)
lodders_table.add_index('element')
wd_abund_table.add_index('name')
bodies_table.add_index('name')

#wd_num_abund_table=Table.read(wd_num_abund_file)
#wd_num_abund_table.add_index('name')

limit_length=0.3 #length of limit error bars on plots
limit_indicator=99. #value above which if the absolute value of the error on a measurement is above it indicates it should be a limit



/Users/BenKaiser/Desktop/radial_velocity_calculations
20220304_WD_SSP_abundances.csv


In [8]:
use_indices=np.where(bodies_table['show']==1)
use_bodies_table=bodies_table[use_indices]
#use_wd_indices=np.where((wd_abund_table['show']==1)and (wd_abund_table['show_geo']==1))
use_wd_indices=np.where(wd_abund_table['show_geo']==1)
use_wd_abund_table=wd_abund_table[use_wd_indices]
use_wd_indices=np.where(use_wd_abund_table['show']==1)
use_wd_abund_table=use_wd_abund_table[use_wd_indices]

In [9]:

t_step=5

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
met_color='#1ca1f2'
met_size=3
ci_size=6
wd_size=10
dp_alpha=0.5
ci_leg_size=9
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
arrow_width=0.03
#arrow_width=0.07

arrow_line=4
figure_text_size=6
default_offset=[0.05,0.00]
annot_line_weight=0.03

show_all_ssobj_names=False

In [10]:
def plot_wd_errorbar(x_coord, el1el2, el1el2_err,name, selected_marker=wd_marker, markersize=wd_size, label='', color='b'):
    if label=='':
        #label=name #commented this on 2021-11-12 to try to keep star points out of legend
        pass 
    else:
        pass
    uplims=False
    lolims=False
    if np.abs(el1el2_err)> limit_indicator:
        if el1el2_err > 0:
            lolims=True
        elif el1el2_err < 0:
            uplims=True
        else:
            print("This shouldn't print el1el2_err")
        el1el2_err= limit_length
    else:
        #no limit indicators are present for the 2 relative abundances input
        pass
    plt.errorbar(x_coord, el1el2, yerr= el1el2_err, uplims=uplims, lolims=lolims,  color=color,marker=selected_marker,  markersize=markersize,linestyle='None')
    plt.errorbar(x_coord,el1el2, label=label, marker=selected_marker, markersize=markersize, color=color,linestyle='None')
    return

In [11]:
def plot_CI_chondrite(x_coord,el1,el2=base_el,label=''):
    el1_val=lodders_table.loc[el1]['A_el']
    el2_val=lodders_table.loc[el2]['A_el']
    el1_err=lodders_table.loc[el1]['A_el_err']
    el2_err=lodders_table.loc[el2]['A_el_err']
    el1el2=el1_val-el2_val
    el1el2_err=np.sqrt(el1_err**2+el2_err**2)
    plt.errorbar(x_coord, el1el2, yerr= el1el2_err,  color=met_color,marker=met_marker,  markersize=ci_size,linestyle='None', label=label)
    return

In [12]:
spt.initiate_science_plot()
spt.start_ApJ_fig(width_cols=2,constrained_layout=True,width_height=[1,0.5])
#plt.figure(figsize=(10.,7.25))
ax=plt.subplot()
for i,name in enumerate(el_list):
    print(i,name)
    edge_spot=i*el_space
    plt.axvline(x=edge_spot, linestyle=':',color='k')
    num_wds=len(use_wd_abund_table)
    tot_objects= num_wds+2 #this is for the future when I include CI Chondrites, continental crust, etc. + number of non-WD points
    object_spacing=el_space/(tot_objects+1)
    start_pos=edge_spot
    el_string=name.lower()+'/'+base_el.lower()
    if i==0:
        CI_label='CI Chond.'
        crust_label='Cont. Crust'
    else:
        CI_label=''
        crust_label=''
    plot_CI_chondrite(start_pos+object_spacing,name,label=CI_label)
    start_pos=start_pos+object_spacing #because we plotted the CI chondrite point
    plt.errorbar(start_pos+object_spacing,crust_table[el_string],color=met_color,marker=r'$\oplus$',markersize=ci_size+2,linestyle='None',label=crust_label)
    start_pos=start_pos+object_spacing #because I also just plotted the continental crust point
    if ssp:
        el_string='ssp_'+el_string
        marker=ssp_marker
        markersize=ci_size
    else:
        marker=wd_marker
        markersize=wd_size
    for j,row in enumerate(use_wd_abund_table):
        object_pos=start_pos+j*object_spacing+object_spacing
        try:
            if i==0:
                label=row['display_name']
            else:
                label=''
            plot_wd_errorbar(object_pos, row[el_string],row[el_string+'_err'],row['display_name'],color=row['plot_color'],selected_marker=marker,markersize=markersize, label=label)
        except KeyError as error:
            print("KeyError:",error)
    #plt.text(edge_spot+0.5*el_space,0,name)
plt.ylabel('log(Z/Ca)')
plt.xlim(0,len(el_list)*el_space)
plt.ylim(-3.5,2)
x_ticks=np.arange(0.5*el_space,len(el_list)*el_space,el_space)
print(x_ticks)
ax.set_xticks(x_ticks)
ax.set_xticklabels(el_list)
plt.legend(loc='best',fontsize=7)
if savefig:
    print(os.getcwd())
    os.chdir(figure_output_dir)
    print(os.getcwd())
    start = time.time()
    print(start)
    time_string=str(start).split('.')[0]
    if ssp:
        plt.savefig("all_elements_ssp"+'_vs_'+base_el+'_'+time_string+'.pdf')#plt.grid(True)
    else:
        plt.savefig("all_elements"+'_vs_'+base_el+'_'+time_string+'.pdf')#plt.grid(True)
    print("Figure saved")
else:
    pass
plt.show()

0 Li
1 Na
2 Mg
3 K
4 Cr
5 Fe
[ 1.  3.  5.  7.  9. 11.]
/Users/BenKaiser/Desktop/radial_velocity_calculations
/Users/BenKaiser/Desktop
1658327539.528814


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)


Figure saved


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:64: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [13]:
print(lodders_table['element'])

element
-------
     Li
     Be
     Na
     Mg
      K
     Ca
     Cr
     Fe


In [14]:
print(matplotlib.__version__)

3.3.4
